# Chapter 11 &mdash; The Historical Importance of Parsing Theory

**Concept 16 of the Chapter 11 decomposition:** *The Historical Importance of Parsing Theory, and Brittle Syntax*

Before precedence grammars, compilers were inscrutable &mdash; and a mistyped period cost a mission.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11-CFG/Concept-History-Of-Parsing/Concept-History-Of-Parsing.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]



import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Parsing theory is one of the clearest cases of theory paying for itself. Before
precedence grammars and the layering of Concept 10, expression parsing was **ad hoc**,
compilers were **inscrutable**, and error messages were worse.

The cautionary tale usually told is an early NASA mission where a **period typed for a
comma** turned a loop header into an assignment &mdash; a one-character edit that the
syntax happily accepted as something else entirely.

The moral is not "be careful". It is that **brittle syntax** &mdash; syntax where a small
edit silently yields a different valid program &mdash; is a design defect. Good grammars
make small errors into **syntax errors**, not into different meanings.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### A brittle syntax, and a robust one

In [ ]:
# Brittle: every two-character string is legal, so ANY typo is still a
# legal -- but different -- program.
Brittle = mkg({'S': ["AB"], 'A': ["a", "b"], 'B': ["a", "b"]})
# Robust: the same payload, wrapped in mandatory delimiters.
Robust  = mkg({'S': ["dLe"], 'L': ["a", "b", "aL", "bL"]})

### Measuring brittleness: how many one-character edits stay legal?

In [ ]:
def one_edits(w, sigma):
    out = set()
    for i in range(len(w)):
        for c in sigma:
            if c != w[i]: out.add(w[:i] + c + w[i+1:])
    return sorted(out)

def brittleness(G, w, upto=8):
    L = set(language(G, upto))
    sigma = sorted(G['Sigma'])
    edits = one_edits(w, sigma)
    still = [e for e in edits if e in L]
    return len(still), len(edits), still

<!-- nav-strip -->

---

&larr;&nbsp;[Ch11&nbsp;15.&nbsp;Mixed Linearity Need Not Be Regular](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11-CFG/Concept-Mixed-Linearity/Concept-Mixed-Linearity.ipynb) &nbsp;&middot;&nbsp; [**Chapter 11** index](https://github.com/ganeshutah/Jove/blob/master/Chapter11-CFG/README.md) &nbsp;&middot;&nbsp; [Ch11&nbsp;17.&nbsp;Combating Inherent Ambiguity: NPDA and DPDA Are Different](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11-CFG/Concept-NPDA-Vs-DPDA/Concept-NPDA-Vs-DPDA.ipynb)&nbsp;&rarr;

---

## 3. Tests

The classic shape of the bug: one character, a different legal program.

In [ ]:
print("intended :  DO 10 I = 1,100     (a loop)")
print("typed    :  DO 10 I = 1.100     (an assignment to the variable DO10I)")
print()
print("Both parse.  Only one is what the author meant.")

Quantified on a toy grammar.

In [ ]:
for name, G, w in [('Brittle', Brittle, 'ab'), ('Robust', Robust, 'dabe')]:
    n, tot, still = brittleness(G, w)
    print("%-8s %-6r : %d of %d one-character edits are STILL legal  %s"
          % (name, w, n, tot, still))

A robust grammar turns most small edits into **syntax errors**.

In [ ]:
n1, t1, _ = brittleness(Brittle, 'ab')
n2, t2, _ = brittleness(Robust, 'dabe')
print("brittle : %.0f%% of edits stay legal" % (100.0 * n1 / t1))
print("robust  : %.0f%% of edits stay legal" % (100.0 * n2 / t2))
assert n1 / t1 > n2 / t2
print("\nDelimiters and keywords are redundancy, and redundancy catches typos.")

What theory bought: precedence, layering, and a *specification* of the language.

In [ ]:
gains = [("before", "ad hoc parsers, undocumented precedence, cryptic errors"),
         ("after",  "a grammar IS the specification; tools derive the parser"),
         ("also",   "ambiguity becomes a checkable property, not a surprise")]
for k, v in gains: print("%-8s %s" % (k, v))

## 4. Exercises


1. Find the actual NASA/Mariner story. Which parts of the common retelling are wrong?
2. Design a syntax where **no** single-character edit yields a different legal program.
3. Which modern languages are most brittle in this sense? Why?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter11-CFG/Concept-History-Of-Parsing')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')